# Evaluation of UQ Methods for MinHash Jaccard Estimates

Demo of 5 UQ methods on short text documents.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])
_pip('loguru')
if 'google.colab' not in sys.modules:
    _pip('numpy', 'matplotlib', 'scipy')
print('Dependencies installed!')

In [ ]:
from loguru import logger
import json, sys, time, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from typing import Dict, Any
from dataclasses import dataclass
warnings.filterwarnings('ignore')
logger.remove()
logger.add(sys.stdout, level='INFO')
print('Imports successful!')

In [ ]:
GITHUB_DATA_URL = 'https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1ba2b3-why-extreme-value-theory-fails-for-minha/main/round-2/evaluation-1/demo/mini_demo_data.json'
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f'GitHub load failed: {e}')
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json')

print('Data loading helper defined!')

In [ ]:
data = load_data()
print(f"Loaded {data['metadata']['num_document_pairs']} document pairs")
print(f"Methods: {', '.join(data['metadata']['methods'])}")

## Configuration

MINIMUM values for fast demo.

In [ ]:
NUM_HASHES = 32
K_SHINGLE = 3
NUM_BOOTSTRAP = 50
CONFIDENCE_LEVELS = [0.95]
MAX_PAIRS = 5
RANDOM_SEED = 42
print('Config set!')

In [ ]:
@dataclass
class UQResult:
    method_name: str
    lower_bound: float
    upper_bound: float
    confidence_level: float
    execution_time: float
    memory_usage_mb: float
    additional_stats: Dict[str, Any]

print('UQResult defined!')

In [ ]:
class MinHashUQEvaluator:
    def __init__(self, num_hashes=32, k_shingle=3, num_bootstrap=50, confidence_levels=None, random_seed=42):
        self.num_hashes = num_hashes
        self.k_shingle = k_shingle
        self.num_bootstrap = num_bootstrap
        self.confidence_levels = confidence_levels or [0.95]
        self.random_seed = random_seed
        np.random.seed(random_seed)

    def compute_minhash_signature(self, text):
        import hashlib
        text = text.lower()
        if len(text) < self.k_shingle:
            shingles = [text]
        else:
            shingles = [text[i:i+self.k_shingle] for i in range(len(text) - self.k_shingle + 1)]
        signature = np.zeros(self.num_hashes)
        for i in range(self.num_hashes):
            min_hash = float('inf')
            for shingle in shingles:
                hash_input = f'{i}_{shingle}'.encode('utf-8')
                hash_value = int(hashlib.md5(hash_input).hexdigest(), 16)
                if hash_value < min_hash:
                    min_hash = hash_value
            signature[i] = min_hash
        return signature

    def estimate_jaccard_from_signatures(self, sig1, sig2):
        return np.mean(sig1 == sig2)

    def compute_true_jaccard(self, text1, text2):
        text1, text2 = text1.lower(), text2.lower()
        if len(text1) < self.k_shingle or len(text2) < self.k_shingle:
            return 0.0
        shingles1 = set(text1[i:i+self.k_shingle] for i in range(len(text1) - self.k_shingle + 1))
        shingles2 = set(text2[i:i+self.k_shingle] for i in range(len(text2) - self.k_shingle + 1))
        intersection = len(shingles1 & shingles2)
        union = len(shingles1 | shingles2)
        return intersection / union if union > 0 else 0.0

In [ ]:
    def evt_gumbel_ci(self, sig1, sig2, confidence_level=0.95):
        jaccard_est = self.estimate_jaccard_from_signatures(sig1, sig2)
        num_matches = np.sum(sig1 == sig2)
        n = self.num_hashes
        p = max(jaccard_est, 1e-10)
        alpha = 1 - confidence_level
        z_alpha = stats.norm.ppf(1 - alpha / 2)
        logit_p = np.log(p / (1 - p)) if p > 0 and p < 1 else 0
        se_logit = 1 / np.sqrt(n * p * (1 - p)) if p > 0 and p < 1 else float('inf')
        logit_lower = logit_p - z_alpha * se_logit
        logit_upper = logit_p + z_alpha * se_logit
        lower = 1 / (1 + np.exp(-logit_lower))
        upper = 1 / (1 + np.exp(-logit_upper))
        return UQResult('EVT-Gumbel', max(0, lower), min(1, upper), confidence_level, 0.0, 0.0, {'jaccard_est': jaccard_est})

    def analytical_binomial_ci(self, sig1, sig2, confidence_level=0.95):
        jaccard_est = self.estimate_jaccard_from_signatures(sig1, sig2)
        num_matches = np.sum(sig1 == sig2)
        n = self.num_hashes
        alpha = 1 - confidence_level
        lower = stats.beta.ppf(alpha / 2, num_matches, n - num_matches + 1)
        upper = stats.beta.ppf(1 - alpha / 2, num_matches + 1, n - num_matches)
        return UQResult('Analytical Binomial', max(0, lower), min(1, upper), confidence_level, 0.0, 0.0, {'jaccard_est': jaccard_est})

In [ ]:
    def evaluate_single_pair(self, text1, text2, true_jaccard, confidence_level=0.95):
        sig1 = self.compute_minhash_signature(text1)
        sig2 = self.compute_minhash_signature(text2)
        jaccard_est = self.estimate_jaccard_from_signatures(sig1, sig2)
        results = {'true_jaccard': true_jaccard, 'jaccard_est': jaccard_est}
        methods = [('evt_gumbel', self.evt_gumbel_ci), ('analytical_binomial', self.analytical_binomial_ci)]
        for method_name, method_func in methods:
            result = method_func(sig1, sig2, confidence_level)
            covered = 1.0 if result.lower_bound <= true_jaccard <= result.upper_bound else 0.0
            results[f'{method_name}_lower'] = result.lower_bound
            results[f'{method_name}_upper'] = result.upper_bound
            results[f'{method_name}_covered'] = covered
            results[f'{method_name}_ci_width'] = result.upper_bound - result.lower_bound
        return results

print('MinHashUQEvaluator defined!')

In [ ]:
evaluator = MinHashUQEvaluator(num_hashes=NUM_HASHES, k_shingle=K_SHINGLE, num_bootstrap=NUM_BOOTSTRAP, random_seed=RANDOM_SEED)
print('Starting evaluation...')
examples = []
for dataset in data['datasets']:
    for example in dataset['examples'][:MAX_PAIRS]:
        examples.append(example)
print(f'Evaluating {len(examples)} document pairs...')
results = []
for i, example in enumerate(examples):
    text1 = example.get('text1', '')
    text2 = example.get('text2', '')
    true_jaccard = example['metadata_true_jaccard']
    result = evaluator.evaluate_single_pair(text1, text2, true_jaccard, confidence_level=CONFIDENCE_LEVELS[0])
    results.append(result)
print(f'Evaluation complete! Processed {len(results)} pairs.')

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Compute coverage
methods = ['evt_gumbel', 'analytical_binomial']
method_labels = ['EVT-Gumbel', 'Analytical Binomial']
coverage_probs = []
for method in methods:
    covered_key = f'{method}_covered'
    coverage_values = [r[covered_key] for r in results if covered_key in r]
    coverage_probs.append(np.mean(coverage_values) if coverage_values else 0.0)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x = np.arange(len(method_labels))
bars = ax.bar(x, coverage_probs, color=['red', 'blue'])
ax.axhline(y=0.95, color='black', linestyle='--', linewidth=2, label='Target (95%)')
ax.set_xlabel('UQ Method')
ax.set_ylabel('Coverage Probability')
ax.set_title('Coverage Probability by Method')
ax.set_xticks(x)
ax.set_xticklabels(method_labels, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary
print('\n' + '='*80)
print('EVALUATION SUMMARY')
print('='*80)
for label, cov in zip(method_labels, coverage_probs):
    status = 'PASS' if abs(cov - 0.95) <= 0.10 else 'FAIL'
    print(f'{label:<25} Coverage: {cov:.4f}  {status}')
